# QC'd objects vs the 2025 objects

Loads the QC'd objects (`qc_passed_mouse.h5ad`, `qc_passed_human.h5ad`) and two sets of 2025 objects, matches nuclei
by lane and 16-base barcode, and reports per lane how many nuclei are shared and how many are in only one object.
Only the cell tables (obs) are read, not the count matrices, so it takes a minute or two. Kernel: `rsc_uv`.

The 2025 references:
- **2025 per-lane QC** (the QC step itself): mouse `2025_03_15_amhr_individual/<lane>_cellbender_doublets_removed.h5ad`;
  human `andy_raw_data/<lane>/cellbender_qc_doublets_labeled.h5ad`, singlets only.
- **2025 final objects**: `20251008_Cevrim_XMens_All_Cells.h5ad` and `20251008_Cevrim_Human_All_Cells.h5ad`. Nuclei
  were removed from these after QC (cell-type annotation, outliers), so many QC'd nuclei are expected to be absent
  from them; the column to watch there is **only 2025**: nuclei of the final objects that the QC did not keep.

As a check that the same underlying data are compared, the last column gives the share of shared nuclei whose
total UMI count is identical in both objects.

In [1]:
import scanpy as sc

In [2]:
adata = sc.read_h5ad('/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_05_qc_verify_pinned/qc_passed_mouse.h5ad')

In [3]:
adata.var

,gene_ids,feature_types,genome,mt,ribo,hb
Xkr4,ENSMUSG00000051951,Gene Expression,mck_genome_gsd_gqd,False,False,False
Gm1992,ENSMUSG00000089699,Gene Expression,mck_genome_gsd_gqd,False,False,False
Gm19938,ENSMUSG00000102331,Gene Expression,mck_genome_gsd_gqd,False,False,False
Gm37381,ENSMUSG00000102343,Gene Expression,mck_genome_gsd_gqd,False,False,False
Rp1,ENSMUSG00000025900,Gene Expression,mck_genome_gsd_gqd,False,False,False
...,...,...,...,...,...,...
ENSMUSG00000094855,ENSMUSG00000094855,Gene Expression,mck_genome_gsd_gqd,False,False,False
ENSMUSG00000095019,ENSMUSG00000095019,Gene Expression,mck_genome_gsd_gqd,False,False,False
ENSMUSG00000095041,ENSMUSG00000095041,Gene Expression,mck_genome_gsd_gqd,False,False,False
GsD,GsD,Gene Expression,mck_genome_gsd_gqd,False,False,False


In [10]:
adata.obs

,sample,species,timepoint,x_um_dbscan,y_um_dbscan,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_ribo,doubletdetection_predicted_doublet,doubletdetection_doublet_score,cell_barcode
GTTTGGACATCATTGG-1-uninduced_mck8a,uninduced_mck8a,mouse,uninduced,NaN,NaN,6619,32178,0.034185,0.320094,False,1.155338e+01,GTTTGGACATCATTGG
CATACAGCACCTCTAC-1-uninduced_mck8a,uninduced_mck8a,mouse,uninduced,4995.393212,1598.647440,6996,31997,0.000000,0.371910,False,3.930269e+00,CATACAGCACCTCTAC
ACCTGAATCCTTCAGC-1-uninduced_mck8a,uninduced_mck8a,mouse,uninduced,2013.199867,3184.682604,6911,31924,0.115900,0.476131,False,1.409876e+01,ACCTGAATCCTTCAGC
GAGCTGCCACTAGAGG-1-uninduced_mck8a,uninduced_mck8a,mouse,uninduced,NaN,NaN,6623,30454,0.072240,0.331648,False,1.155338e+01,GAGCTGCCACTAGAGG
TTCCGTGGTTTCAGAC-1-uninduced_mck8a,uninduced_mck8a,mouse,uninduced,NaN,NaN,6790,30139,0.049769,0.474468,False,1.216255e+01,TTCCGTGGTTTCAGAC
...,...,...,...,...,...,...,...,...,...,...,...,...
TCATCATGTTAAACCC-1-4_dpi_bleeding_mck11a,4_dpi_bleeding_mck11a,mouse,4_dpi_bleeding,8488.505005,6983.248184,715,909,0.110011,3.300330,False,8.090900e-09,TCATCATGTTAAACCC
GCAGCCATCGTGACTA-1-4_dpi_bleeding_mck11a,4_dpi_bleeding_mck11a,mouse,4_dpi_bleeding,6671.402507,8746.412754,635,741,0.000000,2.024291,False,3.699105e-18,GCAGCCATCGTGACTA
GTGACGCCAAGGTCTT-1-4_dpi_bleeding_mck11a,4_dpi_bleeding_mck11a,mouse,4_dpi_bleeding,NaN,NaN,567,880,0.227273,3.409091,False,4.162142e-04,GTGACGCCAAGGTCTT
TCGCTTGAGGCAGCTA-1-4_dpi_bleeding_mck11a,4_dpi_bleeding_mck11a,mouse,4_dpi_bleeding,8268.416797,4954.700326,581,680,0.147059,0.441176,False,4.892898e-03,TCGCTTGAGGCAGCTA


In [4]:
adata_human = sc.read_h5ad('/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_05_qc_verify_pinned/qc_passed_human.h5ad')

In [5]:
adata_human

AnnData object with n_obs × n_vars = 70352 × 36601
    obs: 'sample', 'species', 'timepoint', 'x_um_dbscan', 'y_um_dbscan', 'n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'doubletdetection_predicted_doublet', 'doubletdetection_doublet_score', 'cell_barcode'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb'
    obsm: 'spatial'
    layers: 'counts'

In [15]:
adata_human.obs

,sample,species,timepoint,x_um_dbscan,y_um_dbscan,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_ribo,doubletdetection_predicted_doublet,doubletdetection_doublet_score,cell_barcode
CCTCCTCAGCATGCAG-1-secretory_mck6a,secretory_mck6a,human,secretory,2215.562566,3075.148504,5806,18124,0.000000,0.518649,False,1.903352e+01,CCTCCTCAGCATGCAG
GAATCGTCATCTCATT-1-secretory_mck6a,secretory_mck6a,human,secretory,2148.795492,2394.592036,5546,18166,0.000000,0.258725,False,2.879081e-07,GAATCGTCATCTCATT
GAATCGTCAATATCCG-1-secretory_mck6a,secretory_mck6a,human,secretory,NaN,NaN,6134,18115,0.000000,0.226332,False,3.539821e+00,GAATCGTCAATATCCG
GGCGTCATCTACTGCC-1-secretory_mck6a,secretory_mck6a,human,secretory,4328.199200,3672.392020,5771,17987,0.000000,0.344693,False,2.468696e+00,GGCGTCATCTACTGCC
CTCCCAAAGTGGTTAA-1-secretory_mck6a,secretory_mck6a,human,secretory,1287.346223,3537.689551,5310,17940,0.011148,0.451505,False,2.879081e-07,CTCCCAAAGTGGTTAA
...,...,...,...,...,...,...,...,...,...,...,...,...
TCTCACGGTAGCTCGC-1-menstrual_mck5c,menstrual_mck5c,human,menstrual,1162.573180,3761.082640,549,619,0.969305,1.453958,False,1.197660e-17,TCTCACGGTAGCTCGC
AGAAGCGTCGAACACT-1-menstrual_mck5c,menstrual_mck5c,human,menstrual,3570.077167,4072.407200,562,659,0.000000,0.758725,False,2.501321e-24,AGAAGCGTCGAACACT
TGACTCCTCTCAAAGC-1-menstrual_mck5c,menstrual_mck5c,human,menstrual,NaN,NaN,542,641,0.936037,4.836193,False,1.197660e-17,TGACTCCTCTCAAAGC
TTCTGTAAGGTAGTCA-1-menstrual_mck5c,menstrual_mck5c,human,menstrual,NaN,NaN,551,648,0.000000,0.154321,False,6.696273e-19,TTCTGTAAGGTAGTCA


In [6]:
a = sc.read_h5ad('/n/netscratch/eddy_lab/Everyone/nhilgert/20251008_Cevrim_XMens_All_Cells.h5ad')

In [7]:
a

AnnData object with n_obs × n_vars = 57242 × 33698
    obs: 'cell_barcode', 'cellbender_filtered', 'doubletdetection_doublet_score', 'doubletdetection_predicted_doublet', 'n_counts', 'n_genes_by_counts', 'pct_counts_in_top_20_genes', 'pct_counts_intronic', 'pct_counts_mito', 'pct_counts_ribo', 'sample', 'timepoint', 'total_counts', 'total_counts_mito', 'total_counts_ribo', 'x_um', 'x_um_dbscan', 'y_um', 'y_um_dbscan', 'cell_type_annotations'
    var: 'mito', 'ribo', 'highly_variable', 'highly_variable_rank', 'highly_variable_nbatches'
    uns: 'hvg', 'log1p', 'neighbors_scVI_100', 'rank_genes_groups', 'umap'
    obsm: 'X_scVI', 'X_umap'
    layers: 'log1p', 'raw'
    obsp: 'neighbors_scVI_100_connectivities', 'neighbors_scVI_100_distances'

In [8]:
b = sc.read_h5ad('/n/netscratch/eddy_lab/Everyone/nhilgert/20251008_Cevrim_Human_All_Cells.h5ad')

In [9]:
b

AnnData object with n_obs × n_vars = 64553 × 36601
    obs: 'cell_barcode', 'cellbender_filtered', 'timepoint', 'n_counts', 'x_um', 'y_um', 'x_um_dbscan', 'y_um_dbscan', 'pct_counts_intronic', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'composite_key', 'sample', 'cell_type_annotations'
    var: 'mito', 'ribo', 'highly_variable', 'highly_variable_rank', 'highly_variable_nbatches'
    uns: 'hvg', 'log1p', 'neighbors_scVI_100', 'rank_genes_groups', 'umap'
    obsm: 'X_scVI', 'X_umap'
    layers: 'log1p', 'raw'
    obsp: 'neighbors_scVI_100_connectivities', 'neighbors_scVI_100_distances'

In [12]:
# Uses get_counts() and looks_like_counts() from the previous cell.


def parse_qc_names(qc):
    """Split names like 'AAACCCAAGAAATCCA-1-uninduced_mck8a' into barcode, tag, and sample ID."""
    parts = qc.obs_names.to_series().str.extract(r"^([ACGTN]+)-\d+-(.+)$")
    parts.columns = ["barcode", "tag"]
    assert parts.notna().all().all(), "some QC names don't match BARCODE-N-tag"
    parts["sample"] = parts["tag"].str.rsplit("_", n=1).str[-1]
    return parts


def find_sample_column(full, qc_parts, max_levels=1000):
    """Score each full.obs column by how many of its values match a QC tag or sample ID."""
    targets = {kind: set(qc_parts[kind].str.lower()) for kind in ["tag", "sample"]}
    scores = []
    for col in full.obs.columns:
        if full.obs[col].nunique() > max_levels:
            continue
        values = full.obs[col].astype(str).str.lower()
        for kind, wanted in targets.items():
            scores.append((values.isin(wanted).mean(), col, kind))
    scores = pd.DataFrame(scores, columns=["frac_matched", "column", "matches_qc"])
    return scores.sort_values("frac_matched", ascending=False, ignore_index=True)


def build_keys(qc, full, label, min_frac=0.5):
    """Build 'barcode|sample' keys for both objects so cells can be matched across them."""
    qc_parts = parse_qc_names(qc)

    # Keep only the 16-mer: a trailing -1/-2 on a full-object name may come from
    # obs_names_make_unique rather than the GEM well, so it can't be trusted
    full_barcode = full.obs_names.to_series().str.extract(r"^([ACGTN]+)", expand=False)

    print(f"=== {label}: building match keys ===")
    n_bc = qc_parts["barcode"].isin(set(full_barcode)).sum()
    print(f"QC cells whose 16-mer appears anywhere in full: {n_bc:,} / {qc.n_obs:,}")

    scores = find_sample_column(full, qc_parts)
    print(scores.head(5).to_string(index=False))

    if scores.empty or scores.loc[0, "frac_matched"] < min_frac:
        print("\nNo full.obs column matches the QC sample names. Columns available:")
        print(pd.DataFrame({"n_unique": full.obs.nunique(), "example": full.obs.iloc[0]}).to_string())
        print("QC tags:", sorted(qc_parts["tag"].unique())[:10], "\n")
        return None, None

    col, kind = scores.loc[0, "column"], scores.loc[0, "matches_qc"]
    print(f"-> matching on barcode + full.obs['{col}'] vs QC {kind}\n")
    key_qc = pd.Index(qc_parts["barcode"] + "|" + qc_parts[kind].str.lower())
    key_full = pd.Index(full_barcode + "|" + full.obs[col].astype(str).str.lower())
    return key_qc, key_full


def compare_counts(qc, full, label, key_qc=None, key_full=None,
                   source_qc="auto", source_full="auto", n_show=5):
    key_qc = qc.obs_names if key_qc is None else pd.Index(key_qc)
    key_full = full.obs_names if key_full is None else pd.Index(key_full)

    X_qc, genes_qc, where_qc = get_counts(qc, source_qc)
    X_full, genes_full, where_full = get_counts(full, source_full)

    print(f"=== {label} ===")
    print(f"QC object:   {qc.n_obs:,} cells, counts from {where_qc}")
    print(f"Full object: {full.n_obs:,} cells, counts from {where_full}")
    for name, X in [("QC", X_qc), ("full", X_full)]:
        if not looks_like_counts(X):
            print(f"  WARNING: {name} matrix is not integer-valued; it may be normalized, not raw counts")

    # get_indexer silently picks one row when keys repeat, so duplicates would hide mismatches
    assert key_qc.is_unique, f"{key_qc.duplicated().sum()} duplicate QC keys"
    assert key_full.is_unique, f"{key_full.duplicated().sum()} duplicate full keys"
    assert genes_qc.is_unique and genes_full.is_unique, "duplicate gene names"

    cells = key_qc.intersection(key_full)
    genes = genes_qc.intersection(genes_full)
    print(f"Shared cells: {len(cells):,}  (QC only: {len(key_qc.difference(key_full)):,}, "
          f"full only: {len(key_full.difference(key_qc)):,})")
    print(f"Shared genes: {len(genes):,}  (QC has {len(genes_qc):,}, full has {len(genes_full):,})")
    if len(cells) == 0 or len(genes) == 0:
        print("Nothing to compare.\n")
        return None

    # Align both matrices to the same cell and gene order before comparing
    rows_qc, rows_full = key_qc.get_indexer(cells), key_full.get_indexer(cells)
    cols_qc, cols_full = genes_qc.get_indexer(genes), genes_full.get_indexer(genes)
    A = sp.csr_matrix(X_qc[rows_qc][:, cols_qc], dtype=np.float64)
    B = sp.csr_matrix(X_full[rows_full][:, cols_full], dtype=np.float64)

    diff = A - B
    diff.eliminate_zeros()
    bad_cells = np.unique(diff.nonzero()[0])
    max_abs = abs(diff).max() if diff.nnz else 0.0

    totals = pd.DataFrame(
        {
            "qc_total": np.asarray(A.sum(axis=1)).ravel(),
            "full_total": np.asarray(B.sum(axis=1)).ravel(),
        },
        index=cells,
    )
    totals["diff"] = totals["qc_total"] - totals["full_total"]

    if diff.nnz == 0:
        print("PASS: counts are identical across all shared cells and genes.\n")
    else:
        print(f"FAIL: {diff.nnz:,} entries differ in {len(bad_cells):,} cells "
              f"({len(bad_cells) / len(cells):.2%}); max |diff| = {max_abs:g}")
        print(totals.iloc[bad_cells].head(n_show).to_string(), "\n")

    return {
        "label": label,
        "shared_cells": len(cells),
        "qc_only": len(key_qc.difference(key_full)),
        "full_only": len(key_full.difference(key_qc)),
        "shared_genes": len(genes),
        "differing_entries": diff.nnz,
        "differing_cells": len(bad_cells),
        "max_abs_diff": max_abs,
        "identical": diff.nnz == 0,
    }


pairs = [
    (adata, a, "mouse: qc_passed vs XMens_All_Cells"),
    (adata_human, b, "human: qc_passed vs Human_All_Cells"),
]
results = []
for qc, full, label in pairs:
    key_qc, key_full = build_keys(qc, full, label)
    if key_qc is not None:
        results.append(compare_counts(qc, full, label, key_qc, key_full))
pd.DataFrame([r for r in results if r is not None]).set_index("label")

=== mouse: qc_passed vs XMens_All_Cells: building match keys ===
QC cells whose 16-mer appears anywhere in full: 57,032 / 68,213
 frac_matched                             column matches_qc
     0.294207                             sample        tag
     0.000000                cellbender_filtered        tag
     0.000000                cellbender_filtered     sample
     0.000000 doubletdetection_predicted_doublet        tag
     0.000000 doubletdetection_predicted_doublet     sample

No full.obs column matches the QC sample names. Columns available:
                                    n_unique              example
cell_barcode                           56845     CACACAACAAGATTGA
cellbender_filtered                        1                 True
doubletdetection_doublet_score         23323             2.134217
doubletdetection_predicted_doublet         1                False
n_counts                               13286                 4265
n_genes_by_counts                       5527   

,shared_cells,qc_only,full_only,shared_genes,differing_entries,differing_cells,max_abs_diff,identical
label,,,,,,,,
human: qc_passed vs Human_All_Cells,64283,6069,270,36601,0,0,0.0,True


In [13]:
# Remove underscores and other punctuation so '1dpi_early_mck1' matches '1_dpi_early_mck1'
def norm_label(s):
    return s.astype(str).str.lower().str.replace(r"[^a-z0-9]", "", regex=True)

qc_parts = parse_qc_names(adata)
full_barcode = a.obs_names.to_series().str.extract(r"^([ACGTN]+)", expand=False)

# Every full-object sample should map to a QC tag; anything printed here is unmatched
unmatched = set(norm_label(a.obs["sample"])) - set(norm_label(qc_parts["tag"]))
print("Unmatched full samples:", unmatched or "none")

key_qc = pd.Index(qc_parts["barcode"] + "|" + norm_label(qc_parts["tag"]))
key_full = pd.Index(full_barcode + "|" + norm_label(a.obs["sample"]))
compare_counts(adata, a, "mouse: qc_passed vs XMens_All_Cells", key_qc, key_full)

Unmatched full samples: {'4dpimenstrualmck11a'}
=== mouse: qc_passed vs XMens_All_Cells ===
QC object:   68,213 cells, counts from .layers['counts']
Full object: 57,242 cells, counts from .raw.X
Shared cells: 49,315  (QC only: 18,898, full only: 7,927)
Shared genes: 33,698  (QC has 33,698, full has 33,698)
PASS: counts are identical across all shared cells and genes.



{'label': 'mouse: qc_passed vs XMens_All_Cells',
 'shared_cells': 49315,
 'qc_only': 18898,
 'full_only': 7927,
 'shared_genes': 33698,
 'differing_entries': 0,
 'differing_cells': 0,
 'max_abs_diff': 0.0,
 'identical': True}

In [14]:
import numpy as np
import pandas as pd
import scipy.sparse as sp


def cell_keys(obj):
    """'mck_1:ACGT...' per cell, from obs['sample'] and obs['cell_barcode'] (works for both naming styles)."""
    lane = obj.obs["sample"].astype(str).str.extract(r"mck_?(\d+[a-z]?)$", expand=False)
    barcode = obj.obs["cell_barcode"].astype(str).str.extract(r"([ACGT]{16})", expand=False)
    keys = pd.Index("mck_" + lane + ":" + barcode)
    assert not keys.hasnans and keys.is_unique, "lane or barcode not parsed, or duplicate cells"
    return keys


def counts(obj, layer):
    """The count matrix as CSR: a layer, or X when layer is None."""
    if layer is not None:
        assert layer in obj.layers, f"no layer {layer!r}; layers are {list(obj.layers)}"
    return sp.csr_matrix(obj.X if layer is None else obj.layers[layer])


def compare_counts(new, old, name, new_layer="counts", old_layer="raw"):
    """Counts of the QC'd object (new) vs a 2025 object (old), on the cells and genes both contain."""
    assert new.var_names.is_unique and old.var_names.is_unique, "gene names are not unique"

    # Shared cells and genes, in the same order in both matrices
    k_new, k_old = cell_keys(new), cell_keys(old)
    shared = k_old.intersection(k_new)
    genes = old.var_names.intersection(new.var_names)
    A = counts(new, new_layer)[k_new.get_indexer(shared)][:, new.var_names.get_indexer(genes)].astype(np.float64)
    B = counts(old, old_layer)[k_old.get_indexer(shared)][:, old.var_names.get_indexer(genes)].astype(np.float64)
    if not np.array_equal(B.data, np.round(B.data)):
        print(f"warning: {old_layer!r} of the 2025 object has non-integer values, so it may not be raw counts")

    D = (A - B).tocsr()
    D.eliminate_zeros()
    cell_differs = np.diff(D.indptr) > 0
    totals_equal = np.asarray(A.sum(axis=1)).ravel() == np.asarray(B.sum(axis=1)).ravel()

    print(f"{name}: {len(shared):,} shared cells (QC'd {len(k_new):,}, 2025 {len(k_old):,}); {len(genes):,} shared "
          f"genes (only QC'd {new.n_vars - len(genes):,}, only 2025 {old.n_vars - len(genes):,})")
    print(f"  differing entries: {D.nnz:,} of {B.nnz:,} nonzero; cells with any difference: {cell_differs.sum():,}; "
          f"largest |difference|: {abs(D).max() if D.nnz else 0:g}")
    print("  ALL COUNTS EQUAL on shared cells and genes" if D.nnz == 0 else "  COUNTS DIFFER")

    per_cell = pd.DataFrame({"lane": shared.str.split(":").str[0], "differs": cell_differs,
                             "totals_equal": totals_equal})
    table = per_cell.groupby("lane", sort=False).agg(shared_cells=("differs", "size"),
                                                     cells_with_differences=("differs", "sum"),
                                                     totals_equal=("totals_equal", "sum"))
    table.loc["total"] = table.sum()
    return table


display(compare_counts(adata, a, "mouse"))
display(compare_counts(adata_human, b, "human"))

mouse: 56,885 shared cells (QC'd 68,213, 2025 57,242); 33,698 shared genes (only QC'd 0, only 2025 0)
  differing entries: 0 of 114,789,130 nonzero; cells with any difference: 0; largest |difference|: 0
  ALL COUNTS EQUAL on shared cells and genes


,shared_cells,cells_with_differences,totals_equal
lane,,,
mck_1,3058,0,3058
mck_10a,5698,0,5698
mck_10b,6663,0,6663
mck_11a,7570,0,7570
mck_12a,6880,0,6880
mck_12b,6037,0,6037
mck_15,4219,0,4219
mck_8a,8911,0,8911
mck_8b,7849,0,7849


human: 64,283 shared cells (QC'd 70,352, 2025 64,553); 36,601 shared genes (only QC'd 0, only 2025 0)
  differing entries: 0 of 127,156,325 nonzero; cells with any difference: 0; largest |difference|: 0
  ALL COUNTS EQUAL on shared cells and genes


,shared_cells,cells_with_differences,totals_equal
lane,,,
mck_6c,6895,0,6895
mck_6b,11557,0,11557
mck_6a,11907,0,11907
mck_5c,10512,0,10512
mck_5a,11681,0,11681
mck_5b,11731,0,11731
total,64283,0,64283


In [ ]:
# The QC'd objects to test: the pipeline's (default), or the output folder of a verify_qc.sh run
QC_DIR = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/interim"
# QC_DIR = "/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_05_qc_verify_pinned"

BACKUP = "/n/lab_storage/eddy_lab/Lab/mckinley_backup_2025-12-18/data"
SUBMISSION = "/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions/adatas/submission"
LANES = {
    "mouse": ["mck_8a", "mck_8b", "mck_1", "mck_12a", "mck_12b", "mck_15", "mck_10a", "mck_10b", "mck_11a"],
    "human": ["mck_6a", "mck_6b", "mck_6c", "mck_5a", "mck_5b", "mck_5c"],
}

In [ ]:
import os

import anndata as ad
import numpy as np
import pandas as pd

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")


def read_obs(path):
    """The cell table of an .h5ad, without loading the count matrix."""
    adata = ad.read_h5ad(path, backed="r")
    obs = adata.obs.copy()
    adata.file.close()
    return obs


def lanes_from(sample_names):
    """'1_dpi_early_mck1' or '1dpi_early_mck1' -> 'mck_1'."""
    return ("mck_" + pd.Series(sample_names, dtype=str).str.extract(r"mck_?(\d+[a-z]?)$")[0]).to_numpy()


def barcodes_from(names):
    """The 16-base barcode in a name such as 'AAAC...-1' or 'AAAC...-1-1_dpi_early_mck1'."""
    return pd.Series(names, dtype=str).str.extract(r"([ACGT]{16})")[0].to_numpy()


def keyed(obs, lanes, barcodes):
    """The cell table indexed by 'lane:barcode', the key used to match nuclei between objects."""
    out = obs.copy()
    out.index = pd.Index([f"{lane}:{bc}" for lane, bc in zip(lanes, barcodes)])
    assert not out.index.has_duplicates, "duplicate lane:barcode keys"
    out["lane"] = np.asarray(lanes)
    return out

In [ ]:
# QC'd objects
qcd = {}
for species in LANES:
    obs = read_obs(f"{QC_DIR}/qc_passed_{species}.h5ad")
    qcd[species] = keyed(obs, lanes_from(obs["sample"]), barcodes_from(obs["cell_barcode"]))
    print(f"QC'd {species}: {len(qcd[species]):,} nuclei")

In [ ]:
# 2025 per-lane QC
qc_2025 = {}
parts = []
for lane in LANES["mouse"]:
    obs = read_obs(f"{BACKUP}/2025_03_15_amhr_individual/{lane}_cellbender_doublets_removed.h5ad")
    parts.append(keyed(obs, [lane] * len(obs), barcodes_from(obs.index)))
qc_2025["mouse"] = pd.concat(parts)

parts = []
for lane in LANES["human"]:
    obs = read_obs(f"{BACKUP}/andy_raw_data/{lane}/cellbender_qc_doublets_labeled.h5ad")
    singlet = ~obs["doubletdetection_predicted_doublet"].astype(str).str.lower().eq("true").to_numpy()
    obs = obs[singlet]
    parts.append(keyed(obs, [lane] * len(obs), barcodes_from(obs.index)))
qc_2025["human"] = pd.concat(parts)

for species, obs in qc_2025.items():
    print(f"2025 per-lane QC {species}: {len(obs):,} nuclei")

In [ ]:
# 2025 final objects
final_2025 = {}
for species, name in (("mouse", "XMens"), ("human", "Human")):
    obs = read_obs(f"{SUBMISSION}/20251008_Cevrim_{name}_All_Cells.h5ad")
    final_2025[species] = keyed(obs, lanes_from(obs["sample"]), barcodes_from(obs["cell_barcode"]))
    print(f"2025 final {species}: {len(final_2025[species]):,} nuclei")

In [ ]:
def compare(qcd, ref, lanes):
    """Per lane: nuclei in the QC'd object and the 2025 object, shared, and in only one of them."""
    rows = []
    for lane in lanes:
        a = qcd.index[qcd["lane"] == lane]
        b = ref.index[ref["lane"] == lane]
        shared = a.intersection(b)
        equal = np.isclose(qcd.loc[shared, "total_counts"].astype(float).to_numpy(),
                           ref.loc[shared, "total_counts"].astype(float).to_numpy())
        rows.append({"lane": lane, "QC'd": len(a), "2025": len(b), "shared": len(shared),
                     "only QC'd": len(a) - len(shared), "only 2025": len(b) - len(shared),
                     "% of 2025 kept": 100 * len(shared) / len(b),
                     "total counts equal (%)": 100 * equal.mean() if len(shared) else np.nan})
    table = pd.DataFrame(rows).set_index("lane")
    total = table[["QC'd", "2025", "shared", "only QC'd", "only 2025"]].sum()
    total["% of 2025 kept"] = 100 * total["shared"] / total["2025"]
    shared_all = qcd.index.intersection(ref.index)
    total["total counts equal (%)"] = 100 * np.isclose(
        qcd.loc[shared_all, "total_counts"].astype(float).to_numpy(),
        ref.loc[shared_all, "total_counts"].astype(float).to_numpy()).mean()
    table.loc["total"] = total
    counts = ["QC'd", "2025", "shared", "only QC'd", "only 2025"]
    return table.astype({column: int for column in counts}).round(2)

In [ ]:
results = {}
for species in LANES:
    for label, ref in (("2025 per-lane QC", qc_2025[species]), ("2025 final object", final_2025[species])):
        results[(species, label)] = compare(qcd[species], ref, LANES[species])
        print(f"\n=== {species}: QC'd vs {label}")
        display(results[(species, label)])

**Reading the tables**

- *vs 2025 per-lane QC*: both QCs use the same thresholds on the same CellBender files, so the nuclei in only one
  object come from DoubletDetection's calls, which vary slightly between runs (with the pipeline's own QC objects:
  mouse 67,390 shared of 68,146 in 2025; human 69,900 shared of 70,225).
- *vs 2025 final object*: **only 2025** counts the final objects' nuclei the QC did not keep (with the pipeline's own QC
  objects: 438 of 57,242 mouse, 273 of 64,553 human, 14 of them failing the 2025 thresholds already). **only QC'd**
  is mostly nuclei removed in 2025 after QC.
- *total counts equal (%)*: 100 means the shared nuclei carry identical total UMI counts, i.e. the same data.